In [21]:
!pip install pdfplumber faiss-cpu sentence-transformers transformers

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pdfplumber

def extract_text(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [ ]:
resume_texts = {}

for file_name in uploaded.keys():
    resume_texts[file_name] = extract_text(file_name)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def extract_structured_info(text):
    prompt = f"""
Extract structured information from this resume.

Return ONLY in this format:

Skills: ...
Experience: ...
Projects: ...
Certifications: ...

Resume:
{text[:1500]}
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
    outputs = model.generate(**inputs, max_length=512)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
structured_data = {}

for file_name, text in resume_texts.items():
    print(f"\n📄 {file_name}")
    print("-" * 50)

    structured = extract_structured_info(text)

    structured_data[file_name] = structured

    print(structured)
    print("\n" + "=" * 80)

In [ ]:
cleaned_docs = []

for file_name, content in structured_data.items():
    clean_text = content.replace("\n", " ")

    cleaned_docs.append({
        "file": file_name,
        "text": clean_text
    })

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
import numpy as np

documents = []
embeddings = []

for doc in cleaned_docs:
    emb = embed_model.encode(doc["text"])

    documents.append(doc["file"])
    embeddings.append(emb)

In [ ]:
import faiss

dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings).astype('float32'))

In [ ]:
# def explain_match(jd, resume_text):
#     prompt = f"""
# You are an AI recruiter.

# Job Description:
# {jd}

# Candidate Resume:
# {resume_text}

# Explain:
# - Why this candidate is a good match
# - Matching skills
# - Missing skills

# Keep it short and clear.
# """

#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
#     outputs = model.generate(**inputs, max_length=256)

#     return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
query = input("Enter Job Description: ")

query_embedding = embed_model.encode(query)

D, I = index.search(
    np.array([query_embedding]).astype('float32'),
    k=5
)

print("\n Top Matching resumes:\n")

for rank, (idx, dist) in enumerate(zip(I[0], D[0]), start=1):

    score = 1 / (1 + dist)
    percentage = round(score * 100, 2)

    print(f"Rank {rank}: {documents[idx]}")
    print(f"Match Score: {percentage}%")
    print("-" * 40)

    # explanation = explain_match(query, structured_data[file_name])

    # print("Explanation:")
    # print(explanation)

    # print("-" * 50)